# Pocket chemistry embedding — dimensionality reduction & cluster structure

**Kernel:** `abcfold-npf-notebook` (`envs/notebook.yaml`)

Companion to `gmm_binder_hypothesis.ipynb`. That notebook asked whether the
**redocking energies** form a binder / non-binder split and found they don't —
the one method that *did* separate the GA-importer classes there was the
sequence-only LDA kernel. This notebook looks directly at the
representation that classifier is built on: the **pocket chemistry embedding**.

## The embedding

Exactly the Track A encoding from
`NPF_LDA_kernel/workflow/scripts/analyse_track_a.py` /
`npf_ga_classifier.py` (copied verbatim below, not imported — sibling project):

- each protein has a **35-residue CDD binding-pocket string**
  (`results/ga_classifier/pocket_sites_cdd_msa.tsv`, anchored on 5A2N/4OH3,
  CDD PSSM 340909 Feature 1);
- each residue → its **5 Sandberg Z-scales** (Z1 hydrophilicity, Z2 steric
  bulk, Z3 electronic, Z4 electronegativity/charge, Z5 proline/aromatic
  character); gap/`X` → zero vector;
- so each protein is a point in **35 × 5 = 175-dimensional** physicochemical
  pocket space. This is `X` that the shrinkage-LDA is fit on.

## The question

With the labels **hidden**, does this pocket space have *cluster structure*
at all — and if it does, do the clusters correspond to **GA-import function**,
or merely to **clade** (subfamily / phylogeny)? The `gmm_binder_hypothesis.ipynb`
caveat applies in reverse here: in the strict label every importer is in
NPF1–NPF4 and every non-importer in NPF5–NPF8, so "importers cluster together"
and "NPF1–4 clusters together" are the *same statement* — and telling those
apart is the whole point.

## Label

`bio_label` is the **strict high-confidence split** —
`NPF_LDA_kernel/config/config.yaml`'s `hc_importers` (12) and
`hc_non_importers` (21), the proteins with the strongest experimental evidence.
This is the same label `gmm_binder_hypothesis.ipynb` now uses. **33 of the 53
proteins** in the pocket embedding are in that split (12 importer / 21
non-importer); the other 20 are shown in grey and excluded from every
label-agreement statistic. (An earlier version of this notebook used the
broader `labels.tsv`, 45/53; switching to the strict split drops the
medium-confidence positives NPF2.1/2.6/2.11/2.14/5.1/5.2/5.6/5.7 to grey and
promotes NPF1.1/1.2/4.1/4.2 to importer.)

## Caveats baked in

- **n = 53 proteins, p = 175 features.** Any 2-D projection of 175-D data with
  53 points *will* look like it has structure. Every embedding below is paired
  with a quantitative check (`trustworthiness`, silhouette, and a
  label-permutation kNN-purity test) so we don't over-read a scatter plot.
- **Clade ≡ label here.** In the strict split every HC importer is in NPF1–NPF4
  and every HC non-importer is in NPF5–NPF8 — a perfectly clean subfamily
  partition. So clustering that recovers clade will *automatically* look like it
  recovers the label; the permutation test in §7 (label purity restricted to
  `clade != NPF2`) is the only thing that can tell them apart, and even that is
  thin once the non-NPF2 importers are just NPF1.1/1.2, NPF3.1, NPF4.1/4.2.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA
from sklearn.manifold import TSNE, MDS, trustworthiness
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score, roc_auc_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import LeaveOneOut

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)

ROOT = Path("..").resolve()
LDA_KERNEL = ROOT.parent / "NPF_LDA_kernel" / "results" / "ga_classifier"
RNG = 0
np.random.seed(RNG)

# --- Z-scales (Sandberg et al. 1998) -- identical table to analyse_track_a.py
ZSCALES = {
    "A": (0.24, -2.32, 0.60, -0.14, 1.30), "R": (3.52, 2.50, -3.50, 1.99, -0.17),
    "N": (3.05, 1.62, 1.04, -1.15, 1.61),  "D": (3.98, 0.93, 1.93, -2.46, 0.75),
    "C": (0.84, -1.67, 3.71, 0.18, -2.65), "Q": (1.75, 0.50, -1.44, -1.34, 0.66),
    "E": (3.11, 0.26, -0.11, -3.04, -0.25), "G": (2.05, -4.06, 0.36, -0.82, -0.38),
    "H": (2.47, 1.95, 0.26, 3.90, 0.09),   "I": (-3.89, -1.73, -1.71, -0.84, 0.26),
    "L": (-4.28, -1.30, -1.49, -0.72, 0.84), "K": (2.29, 0.89, -2.49, 1.49, 0.31),
    "M": (-2.85, -0.22, 0.47, 1.94, -0.98), "F": (-4.22, 1.94, 1.06, 0.54, -0.62),
    "P": (-1.66, 0.27, 1.84, 0.70, 2.00),  "S": (2.39, -1.07, 1.15, -1.39, 0.67),
    "T": (0.75, -2.18, -1.12, -1.46, -0.40), "V": (-2.59, -2.64, -1.54, -0.85, -0.02),
    "W": (-4.36, 3.94, 0.59, 3.44, -1.59), "Y": (-2.54, 2.44, 0.43, 0.04, -1.47),
    "X": (0.0, 0.0, 0.0, 0.0, 0.0),
}
ZDIM = 5
Z_NAMES = ["Z1_hydrophil", "Z2_steric", "Z3_electronic", "Z4_electroneg", "Z5_proline"]


def encode(pocket: str) -> np.ndarray:
    out = []
    for ch in pocket.upper():
        out.extend(ZSCALES.get(ch, (0.0,) * ZDIM))
    return np.asarray(out, dtype=float)


LABEL_COLORS = {"importer": "#2ca02c", "non-importer": "#7f7f7f", "unlabelled": "#d9d9d9"}

## 1. Build the 53 × 175 pocket embedding

In [ ]:
import json

sites = {}
for i, line in enumerate((LDA_KERNEL / "pocket_sites_cdd_msa.tsv").read_text().splitlines()):
    if i == 0:
        continue
    name, pocket = line.split("\t")
    sites[name] = pocket

# --- strict high-confidence split (NPF_LDA_kernel config.yaml) ---
HC_IMPORTERS = {
    "NPF3.1", "NPF4.1", "NPF2.12", "NPF2.13", "NPF2.10", "NPF2.5",
    "NPF2.7", "NPF2.3", "NPF2.4", "NPF4.2", "NPF1.1", "NPF1.2",
}
HC_NON_IMPORTERS = {
    "NPF8.1", "NPF8.2", "NPF8.3", "NPF8.4", "NPF8.5", "NPF6.1", "NPF6.2",
    "NPF6.3", "NPF6.4", "NPF7.1", "NPF7.2", "NPF7.3", "NPF5.8", "NPF5.9",
    "NPF5.10", "NPF5.11", "NPF5.12", "NPF5.13", "NPF5.14", "NPF5.15", "NPF5.16",
}
hc_label = {**{p: 1 for p in HC_IMPORTERS}, **{p: 0 for p in HC_NON_IMPORTERS}}

# original assay label (labels.tsv), kept only as a reference column
assay = {}
for line in (LDA_KERNEL / "labels.tsv").read_text().splitlines():
    n, v = line.split()
    assay[n.rsplit("_", 1)[0]] = int(v)

lda_seq_score = {k.rsplit("_", 1)[0]: v
                 for k, v in json.load(open(LDA_KERNEL / "track_b_spectrum_k2.json"))["scores"].items()}

names = sorted(sites)
meta = pd.DataFrame({"name": names})
meta["npf"] = meta["name"].str.rsplit("_", n=1).str[0]
meta["clade"] = meta["npf"].str.split(".").str[0]
meta["bio_label"] = meta["npf"].map(hc_label)                      # 1 / 0 / NaN  (strict HC split)
meta["label_str"] = meta["bio_label"].map({1: "importer", 0: "non-importer"}).fillna("unlabelled")
meta["assay_label"] = meta["npf"].map(assay)                       # labels.tsv, reference only
meta["lda_seq_score"] = meta["npf"].map(lda_seq_score)
meta["pocket"] = meta["name"].map(sites)

X_raw = np.vstack([encode(sites[n]) for n in names])               # (53, 175)
FEATURE_NAMES = [f"p{pos+1:02d}_{Z_NAMES[z]}" for pos in range(35) for z in range(ZDIM)]

# per-feature standardization (z-scales differ in scale across positions);
# columns that are constant (e.g. a position that never varies) -> drop.
nonconst = X_raw.std(axis=0) > 1e-9
Xs = StandardScaler().fit_transform(X_raw[:, nonconst])
print(f"embedding: {X_raw.shape}  ->  {Xs.shape} after dropping {(~nonconst).sum()} constant columns")
print(f"labelled: {meta.bio_label.notna().sum()}/53  (importer={int(meta.bio_label.sum())})")
print(meta.groupby(['clade', 'label_str']).size().unstack(fill_value=0))

## 2. Raw structure — hierarchical clustering & distance heatmap

Before any projection: Ward-linkage dendrogram on the standardized 175-D
vectors, and the protein–protein Euclidean distance matrix reordered by that
linkage. Leaf colours = clade. If the pocket space is organised by anything,
the dendrogram shows it here without a 2-D bottleneck.

In [ ]:
from scipy.cluster.hierarchy import leaves_list

D = squareform(pdist(Xs, metric="euclidean"))
Z = linkage(Xs, method="ward")
short = [n.split("_")[0] for n in names]

dend = ff.create_dendrogram(Xs, orientation="bottom", linkagefun=lambda _: Z, labels=short)
dend.update_layout(title="Ward linkage on standardized 175-D pocket embedding",
                   height=440, xaxis_tickangle=-60, margin=dict(b=120))
dend.show()

leaf_idx = leaves_list(Z)
heat_names = [short[i] for i in leaf_idx]
fig = px.imshow(D[np.ix_(leaf_idx, leaf_idx)], x=heat_names, y=heat_names,
                color_continuous_scale="Viridis_r", aspect="equal",
                title="pairwise Euclidean distance in pocket space (Ward-leaf order)")
fig.update_layout(height=640, xaxis_tickangle=-60)
fig.show()

# how well does a flat cut of this tree match clade vs label?
m_lab0 = meta["bio_label"].notna().values
print("flat cuts of the Ward tree:")
for k in (2, 3, 4, 5, 6, 8):
    cl = fcluster(Z, k, criterion="maxclust")
    ari_clade = adjusted_rand_score(meta["clade"], cl)
    ari_label = adjusted_rand_score(meta.loc[m_lab0, "bio_label"], cl[m_lab0])
    print(f"  k={k}:  ARI vs clade = {ari_clade:+.3f}   ARI vs label = {ari_label:+.3f}")

## 3. PCA — linear, interpretable

Scree first (how many components carry real variance), then PC1–PC2 / PC1–PC3
coloured by HC label, marker symbol by clade.

In [ ]:
pca = PCA(random_state=RNG).fit(Xs)
evr = pca.explained_variance_ratio_
cum = np.cumsum(evr)
n80 = int(np.argmax(cum >= 0.80) + 1)
n90 = int(np.argmax(cum >= 0.90) + 1)

fig = go.Figure()
fig.add_bar(y=evr[:25], name="per PC")
fig.add_scatter(y=cum[:25], name="cumulative", yaxis="y2", mode="lines+markers")
fig.update_layout(title=f"PCA scree — {n80} PCs for 80%, {n90} PCs for 90% variance",
                  height=360, yaxis=dict(title="explained var ratio"),
                  yaxis2=dict(title="cumulative", overlaying="y", side="right", range=[0, 1]))
fig.show()

P = pca.transform(Xs)
pcs = pd.DataFrame(P[:, :6], columns=[f"PC{i+1}" for i in range(6)])
pcs = pd.concat([meta[["name", "npf", "clade", "label_str", "lda_seq_score"]], pcs], axis=1)

for x, y in [("PC1", "PC2"), ("PC1", "PC3"), ("PC2", "PC3")]:
    fig = px.scatter(pcs, x=x, y=y, color="label_str", symbol="clade", hover_name="npf",
                     color_discrete_map=LABEL_COLORS,
                     title=f"pocket-chemistry PCA — {x} vs {y}")
    fig.update_traces(marker_size=12, marker_line_width=1)
    fig.update_layout(height=520)
    fig.show()

## 4. Non-linear projections — t-SNE, UMAP, MDS, kernel PCA

t-SNE and UMAP are run on the top `n90` PCs (denoise first, standard practice).
With n=53 only small neighbourhood sizes are meaningful, so both get a small
sweep (t-SNE perplexity, UMAP `n_neighbors`). Every panel reports
`trustworthiness` (1.0 = local neighbourhoods preserved perfectly; < 0.9 = read
the layout with suspicion). The UMAP panels are coloured by **clade**
(HC label shown as marker symbol) — the §5 clustering numbers say the
structure here is subfamily, and UMAP tends to render that most sharply.

In [ ]:
Xtop = P[:, :max(n90, 5)]

def scatter_embed(emb, title, tw, color="label_str"):
    d = meta[["npf", "clade", "label_str"]].copy()
    d["e1"], d["e2"] = emb[:, 0], emb[:, 1]
    kw = dict(color_discrete_map=LABEL_COLORS) if color == "label_str" else {}
    symbol = "clade" if color == "label_str" else "label_str"
    fig = px.scatter(d, x="e1", y="e2", color=color, symbol=symbol, hover_name="npf",
                     title=f"{title}   (trustworthiness={tw:.3f})", **kw)
    fig.update_traces(marker_size=12, marker_line_width=1)
    fig.update_layout(height=480, xaxis_title="", yaxis_title="")
    fig.show()

for perp in (5, 10, 15, 30):
    ts = TSNE(n_components=2, perplexity=perp, init="pca", random_state=RNG,
              learning_rate="auto").fit_transform(Xtop)
    tw = trustworthiness(Xtop, ts, n_neighbors=min(10, perp))
    scatter_embed(ts, f"t-SNE on top {Xtop.shape[1]} PCs — perplexity {perp}", tw)

In [ ]:
# --- UMAP, coloured by clade -------------------------------------------------
import umap

for nn in (5, 10, 15):
    um = umap.UMAP(n_neighbors=nn, min_dist=0.1, n_components=2,
                   metric="euclidean", random_state=RNG).fit_transform(Xtop)
    tw = trustworthiness(Xtop, um, n_neighbors=min(10, nn))
    scatter_embed(um, f"UMAP on top {Xtop.shape[1]} PCs — n_neighbors={nn}  (colour = clade)",
                  tw, color="clade")

# same UMAP (n_neighbors=15) once more, coloured by HC label for continuity
um15 = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=RNG).fit_transform(Xtop)
scatter_embed(um15, "UMAP n_neighbors=15  (colour = HC label)",
              trustworthiness(Xtop, um15, n_neighbors=10))

# does the UMAP layout recover clade / label? (KMeans on the 2-D embedding)
for name, groups in [("clade", meta["clade"].astype("category").cat.codes.values),
                     ("label", meta.loc[m_lab0, "bio_label"].values)]:
    km = KMeans(len(np.unique(groups)), n_init=20, random_state=RNG).fit_predict(um15)
    g = km if name == "clade" else km[m_lab0]
    print(f"UMAP(n_neighbors=15) KMeans ARI vs {name} = {adjusted_rand_score(groups, g):+.3f}")

In [ ]:
mds = MDS(n_components=2, random_state=RNG, normalized_stress="auto",
          init="random", n_init=4).fit_transform(Xs)
scatter_embed(mds, "metric MDS on standardized 175-D", trustworthiness(Xs, mds, n_neighbors=10))

kpca = KernelPCA(n_components=2, kernel="rbf",
                 gamma=1.0 / Xs.shape[1], random_state=RNG).fit_transform(Xs)
scatter_embed(kpca, "kernel PCA (RBF)", trustworthiness(Xs, kpca, n_neighbors=10))

## 5. Is there *cluster* structure — and does it track clade or function?

Unsupervised clustering on the top-`n90` PC representation: KMeans and a
diagonal-covariance GMM over `k = 2..8` (silhouette / BIC), plus Ward and
HDBSCAN. For the best solution of each, ARI against **clade** and against the
**assay label**. Expectation from §1: whatever structure exists tracks clade.

In [ ]:
import hdbscan

m_lab = meta["bio_label"].notna().values
y_lab = meta.loc[m_lab, "bio_label"].values
clade_codes = meta["clade"].astype("category").cat.codes.values

rows = []
for k in range(2, 9):
    km = KMeans(k, n_init=20, random_state=RNG).fit_predict(Xtop)
    gm = GaussianMixture(k, covariance_type="diag", n_init=10, random_state=RNG).fit(Xtop)
    gml = gm.predict(Xtop)
    rows.append({
        "k": k,
        "kmeans_sil": silhouette_score(Xtop, km),
        "kmeans_ARI_clade": adjusted_rand_score(clade_codes, km),
        "kmeans_ARI_label": adjusted_rand_score(y_lab, km[m_lab]),
        "gmm_bic": gm.bic(Xtop),
        "gmm_ARI_clade": adjusted_rand_score(clade_codes, gml),
        "gmm_ARI_label": adjusted_rand_score(y_lab, gml[m_lab]),
    })
clust = pd.DataFrame(rows)
print(clust.round(3).to_string(index=False))

for k in (2, 3, 4):
    w = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(Xtop)
    print(f"Ward k={k}:  ARI clade={adjusted_rand_score(clade_codes, w):+.3f}   "
          f"ARI label={adjusted_rand_score(y_lab, w[m_lab]):+.3f}")

hdb = hdbscan.HDBSCAN(min_cluster_size=3, min_samples=1).fit_predict(Xtop)
n_noise = int((hdb == -1).sum())
print(f"\nHDBSCAN(min_cluster_size=3): {len(set(hdb)) - (1 if n_noise else 0)} clusters, "
      f"{n_noise} noise pts")
if len(set(hdb)) > 1:
    print(f"  ARI clade={adjusted_rand_score(clade_codes, hdb):+.3f}   "
          f"ARI label={adjusted_rand_score(y_lab, hdb[m_lab]):+.3f}")
print(pd.crosstab(meta['clade'], pd.Series(hdb, name='hdbscan')))

## 6. Supervised reference — the LDA axis, and is the label direction even in the top PCs?

Fit the same shrinkage-LDA (`solver="lsqr", shrinkage="auto"`) that Track A
uses, on the 33 HC-labelled proteins. Show its 1-D discriminant, its LOO AUC,
and **how much of the LDA direction lives in the leading PCs** — i.e. would an
unsupervised PCA ever have found it. (On this near-perfectly clade-separated
33-protein set the in-sample fit is trivially perfect and the LOO AUC is high;
what matters is the PC-overlap bar chart — a high LOO AUC with near-zero
overlap means the signal is a low-variance direction DR cannot see.)

In [ ]:
Xl = Xs[m_lab]
lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto").fit(Xl, y_lab)
disc = lda.decision_function(Xs)                       # project all 53
meta["lda_pocket_disc"] = disc

# LOO AUC of the pocket LDA
oof = np.zeros(len(y_lab))
for tr, te in LeaveOneOut().split(Xl):
    f = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto").fit(Xl[tr], y_lab[tr])
    oof[te] = f.decision_function(Xl[te])
print(f"pocket-LDA  in-sample AUC = {roc_auc_score(y_lab, lda.decision_function(Xl)):.3f}   "
      f"LOO AUC = {roc_auc_score(y_lab, oof):.3f}")
print(f"sequence kernel LOO AUC on this HC split (hc/summary.tsv): 0.996 (spectrum k2) / 1.00 (k3)")

# cosine of LDA weight vector with each PC axis
w = lda.coef_.ravel() / np.linalg.norm(lda.coef_.ravel())
cos_pc = pca.components_ @ w
fig = go.Figure(go.Bar(x=[f"PC{i+1}" for i in range(15)], y=np.abs(cos_pc[:15])))
fig.update_layout(title="|cos(angle)| between the pocket-LDA direction and each PC "
                        f"(first PC with |cos|>0.3: PC{int(np.argmax(np.abs(cos_pc) > 0.3) + 1)})",
                  height=340, yaxis_title="|cosine|")
fig.show()

d = meta.copy()
fig = px.strip(d, x="label_str", y="lda_pocket_disc", color="label_str", hover_name="npf",
               color_discrete_map=LABEL_COLORS,
               title="pocket-LDA discriminant (fit on labelled, applied to all 53)")
fig.update_traces(marker_size=10, jitter=0.35)
fig.update_layout(height=420, showlegend=False)
fig.show()

fig = px.scatter(pcs.assign(lda_disc=disc), x="PC1", y="PC2", color=disc,
                 symbol=meta["label_str"], hover_name="npf", color_continuous_scale="RdBu_r",
                 title="PCA plane coloured by the (supervised) pocket-LDA discriminant")
fig.update_traces(marker_size=12, marker_line_width=1)
fig.update_layout(height=520)
fig.show()

## 7. Permutation test — is the importer grouping tighter than chance?

With n=53, p=175, apparent separation can be an artifact. Two kNN-purity
statistics in the top-`n90` PC space: for each protein, the fraction of its
`k` nearest neighbours sharing its **label** (labelled subset only) and its
**clade** (all 53). Compare the observed mean to 2000 random label / clade
permutations → empirical p.

**On the strict HC split the label and the clade are the same partition**
(importers = NPF1–4, non-importers = NPF5–8), so a significant label result
here is *expected* and says nothing beyond "different subfamilies have
different pockets". The `clade != NPF2` row was meant to isolate a
function-not-phylogeny signal, but on this label set even that subset is
100% clade-split, so it can't. The clade purity test (all 53) is the one that
still means what it says.

In [ ]:
from sklearn.neighbors import NearestNeighbors

def knn_purity(emb, groups, k=5):
    nn = NearestNeighbors(n_neighbors=k + 1).fit(emb)
    idx = nn.kneighbors(return_distance=False)[:, 1:]
    return np.mean([(groups[nbrs] == groups[i]).mean() for i, nbrs in enumerate(idx)])


def perm_p(emb, groups, k=5, n=2000):
    obs = knn_purity(emb, groups, k)
    null = np.array([knn_purity(emb, np.random.permutation(groups), k) for _ in range(n)])
    return obs, null.mean(), (np.sum(null >= obs) + 1) / (n + 1)

clade_all = meta["clade"].values
obs_c, exp_c, p_c = perm_p(Xtop, clade_all, k=5)
print(f"clade  kNN-purity (all 53):  observed={obs_c:.3f}  null~{exp_c:.3f}  p={p_c:.4f}")

obs_l, exp_l, p_l = perm_p(Xtop[m_lab], meta.loc[m_lab, 'bio_label'].values, k=5)
print(f"label  kNN-purity ({int(m_lab.sum())} lab):  observed={obs_l:.3f}  null~{exp_l:.3f}  p={p_l:.4f}")

# label purity *within* the non-NPF2 subset. NB: on the strict HC split this
# does NOT actually break the clade confound -- the non-NPF2 importers are only
# NPF1.1/1.2, NPF3.1, NPF4.1/4.2, so this is still "NPF1/3/4 pockets vs
# NPF5-8 pockets", i.e. clade by another name. Reported for continuity with the
# broad-label version of this notebook, where NPF5 held both classes.
non_npf2 = m_lab & (meta["clade"] != "NPF2").values
if non_npf2.sum() > 8 and 0 < meta.loc[non_npf2, "bio_label"].sum() < non_npf2.sum():
    o, e, p = perm_p(Xtop[non_npf2], meta.loc[non_npf2, "bio_label"].values, k=3)
    print(f"label  kNN-purity (labelled, clade != NPF2; n={int(non_npf2.sum())}, "
          f"{int(meta.loc[non_npf2, 'bio_label'].sum())} importer):  "
          f"observed={o:.3f}  null~{e:.3f}  p={p:.4f}   <- still clade-confounded, see comment")

## 8. Reading of the result

*Decision guide:*

| observation | reading |
|---|---|
| dendrogram / clustering ARI is high vs **clade** and tracks **label** only as far as label tracks clade | pocket space is organised by **phylogeny**; "binder cluster" and "subfamily cluster" are indistinguishable. |
| clade kNN-purity `p` ≪ 0.05 | pocket chemistry strongly encodes subfamily (the one test that isn't circular on the strict label). |
| label kNN-purity `p` significant — expected here, since label ≡ NPF1–4 / NPF5–8 | says nothing beyond "different subfamilies have different pockets". |
| pocket-LDA LOO AUC ≫ 0.5 but the LDA direction has near-zero cosine with the top ~10 PCs | the discriminative direction is a **low-variance** direction — unsupervised DR will never surface it; only a supervised fit finds it. |
| every embedding has `trustworthiness` < 0.9 | don't interpret the 2-D layouts literally; rely on the numeric tests. |

### Observed outcome — strict HC split (re-run 2026-08-28)

**The pocket chemistry embedding has clear cluster structure, and on the strict
HC label it also "separates importers" — but only because that label *is* a
clean NPF1–4 / NPF5–8 subfamily split. It remains a phylogeny signal; the
strict label just makes phylogeny and function impossible to tell apart.**

- **Clustering.** Ward flat cuts: ARI vs clade climbs to **+0.55** (k=8); ARI
  vs label now also rises (to **+0.28** at k=8, **+0.44** for KMeans k=6,
  **+0.57** for GMM k=5) — higher than on the broad label, precisely because
  the HC label coincides with subfamily. HDBSCAN still finds 8 near-pure
  subfamily clusters (ARI vs clade **+0.44**, vs label +0.14). **UMAP** (§4):
  KMeans on the UMAP plane gives ARI **+0.50 vs clade / +0.30 vs label** — same
  ordering, label only trails clade.
- **Permutation test.** Clade kNN-purity (all 53): observed 0.517 vs null
  0.184, **p = 0.0005** — the non-circular result: pocket space strongly
  encodes subfamily. Label kNN-purity (33): 0.770 vs 0.523, **p = 0.0005**;
  and unlike the broad-label run it *stays* significant on `clade != NPF2`
  (n = 26, obs 0.808 vs null 0.679, **p = 0.021**) — **but that subset is now
  itself 100% clade-split** (5 importers, all NPF1/3/4; 21 non-importers, all
  NPF5–8), so it is not the phylogeny-free test it was designed to be. There is
  no clade in the strict split that contains both classes, so no test here can
  separate "importer pocket" from "NPF1–4 pocket".
- **Supervised fit.** Pocket-LDA LOO AUC is now **1.00** (was 0.82 on the broad
  label), matching the sequence kernel's ~1.0 on this split — expected when the
  classes are a clean subfamily partition. Still check §6's PC-overlap bar: a
  high LOO AUC with near-zero cosine to the leading PCs means the signal sits
  on a low-variance axis unsupervised DR cannot reach.

### What this means for the contamination hypothesis

- **Neither label choice rescues the pocket embedding as a discovery tool.** On
  the broad label the importer grouping vanished once NPF2 was removed; on the
  strict label the grouping is real but inseparable from subfamily. Either way,
  "find the binder cluster in pocket space" is not a route to contamination
  candidates.
- The only phylogeny-free way to use this representation is as **residuals**:
  regress clade (or the phylogenetic tree) out of the Z-scale features, then
  test whether any leftover axis correlates with the label or with
  `gmm_binder_hypothesis.ipynb`'s disagreement list.
- Proteins whose **pocket-LDA discriminant disagrees with their label** (§6
  strip plot) are supervised-model leads, not findings — and on a
  perfectly-separated 33-protein fit there will be very few. Cross-check any
  against the sequence-kernel score and the `assay_label` column.
- If the PI can supply quantitative transport rates, redo §6 as a regression on
  the Z-scale features and read the loadings by pocket position
  (`hc_lda_loadings.tsv` style) for which positions carry the graded signal.